# Phase 3: GPU Optimized Implementation
**CSC14120 - Parallel Programming**

## Optimizations Applied (vs Phase 2 Naive):

### Category 1: Memory Optimization
1. **Shared Memory Tiling (#1)** - Tiled Conv2D forward for all 5 conv layers (main speedup)
2. **Pinned Memory (#5)** - `cudaMallocHost` for faster CPU-GPU transfers

### Category 2: Kernel-Level Optimization
3. **Loop Unrolling (#10)** - `#pragma unroll` in conv/pool kernels
4. **Vectorized Memory Access (#11)** - `float4` ReLU forward/backward
5. **Optimized Block Dimensions (#12)** - 16x16 thread blocks for 2D kernels

(Backward Conv vẫn dùng kernel Phase 2 để đảm bảo ổn định, dễ debug; Phase 3 chỉ thay thế **Conv forward** bằng bản tiled + vectorized ReLU.)

## Huong dan:
1. Zip thu muc project (khong bao gom data/)
2. Upload file zip len Colab
3. Chay tat ca cells

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
from google.colab import files
import zipfile, os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        %cd {root}
        break
!ls

In [ ]:
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Done!')

In [ ]:
# Build Phase 3 (GPU Optimized) - có flag USE_OPTIMIZED_KERNELS
!nvcc -O2 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr -DUSE_OPTIMIZED_KERNELS -Iinclude \
    -o gpu_train_opt src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp
print('Build complete!')

In [ ]:
# Train
!./gpu_train_opt --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase3.csv --log-txt phase3.txt --save-weights phase3.weights

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase3.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'r-o'); ax1.set_title('Loss'); ax1.grid(True)
ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'orange', marker='o'); ax2.set_title('Time (s)'); ax2.grid(True)
plt.tight_layout(); plt.show()

print(f"Best Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Avg Time: {ep['epoch_time_sec'].mean():.2f}s/epoch")
print(f"Total: {ep['epoch_time_sec'].sum():.2f}s")

In [ ]:
files.download('phase3.csv')
files.download('phase3.txt')
files.download('phase3.weights')